# MLflow Prompt Registry
In this notebook, we will demonstrate how to make use of MLflow's prompt registry functionality.

## 0: Notebook Setup
We'll set ourselves up for success by importing our Python dependencies and setting up the MLflow connection.

In [9]:
# Importing the necessary Python libraries
import mlflow
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List

In [2]:
# Setting the base URL for our MLflow instance
MLFLOW_BASE_URL = 'http://127.0.0.1:5000'

# Setting the OpenAI 5.4 nano endpoint from MLflow
OPENAI_GPT_5_4_NANO_ENDPOINT = 'openai-gpt-5.4-nano'

# Instantiating the OpenAI SDK client
openai_client = OpenAI(
    base_url = f'{MLFLOW_BASE_URL}/gateway/openai/v1',
    api_key = 'dummy'
)

# Pointing to the MLflow tracking server
mlflow.set_tracking_uri(MLFLOW_BASE_URL)


In [ ]:
# Setting our summarization text prompt
SUMMARIZATION_PROMPT = '''
Summarize the following text in {{ num_sentences }}.

Here is the text to summarize:
{{ text }}
'''

# Setting our starter chat prompt
SUMMARIZATION_CHAT_PROMPT = [
    {
        'role': 'system',
        'content': 'Summarize content you are provided with in {{ num_sentences }} sentences.'
    },
    {
        'role': 'user',
        'content': 'Here is the text to summarize: {{ sentences }}'
    }
]

# Creating a dummy write up of MLflow to later summarize
MLFLOW_WRITE_UP = '''
MLflow is an open source platform for managing the machine learning and generative AI lifecycle. It helps teams organize experiments, track model behavior, package code, register models, and move machine learning systems from development into production. Instead of relying on scattered notebooks, ad hoc files, and manual handoffs, MLflow provides a common system of record for models, prompts, evaluations, and deployment artifacts.

One of MLflow’s core strengths is experiment tracking. Data scientists and machine learning engineers can log parameters, metrics, artifacts, datasets, and model outputs during training or evaluation runs. This makes it easier to compare different approaches, reproduce past results, and understand why one model performed better than another. The MLflow UI provides a central place to inspect runs, compare metrics, review artifacts, and share results with other team members.

MLflow also supports model packaging and model management. With MLflow Models, teams can save models in a standardized format that includes the model artifact, environment information, and inference interface. This makes it easier to move a model between local development, batch scoring, real-time serving, and other deployment targets. The MLflow Model Registry adds governance by allowing teams to register versions of a model, track its stage, document changes, and coordinate promotion toward production.

For generative AI use cases, MLflow has expanded beyond traditional model tracking. Teams can use MLflow to track prompts, evaluate LLM responses, capture traces, compare different providers or models, and monitor how applications behave across development and production workflows. This is especially useful when building applications that depend on prompt templates, retrieval systems, agents, or external model APIs. By recording inputs, outputs, metadata, and evaluation results, MLflow helps teams understand not only whether an application works, but why it works.

MLflow is valuable because it brings structure and repeatability to work that can otherwise become difficult to manage. Machine learning and AI projects often involve many experiments, changing datasets, multiple contributors, and evolving deployment requirements. MLflow gives teams a shared framework for tracking decisions, reviewing results, and improving systems over time. Whether a team is training a classical machine learning model, evaluating a large language model, or building a GenAI application, MLflow helps make the process more observable, reproducible, and collaborative.
'''

# Creating a dummy sentiment analysis prompt
SENTIMENT_ANALYSIS_PROMPT = '''
Classify the sentiment. Answer 'positive' or 'negative' or 'neutral'.

Here are the sentences to classify:

{{ sentences }}
'''

# Creating sentiment examples
SENTIMENT_EXAMPLES = [
    {
      "text": "MLflow made it so much easier to keep track of all my machine learning experiments.",
      "sentiment": "positive"
    },
    {
      "text": "I love how MLflow lets me compare model runs without digging through old notebooks.",
      "sentiment": "positive"
    },
    {
      "text": "Registering models in MLflow has made deployments much more organized.",
      "sentiment": "positive"
    },
    {
      "text": "Our team adopted MLflow, and collaboration has improved dramatically.",
      "sentiment": "positive"
    },
    {
      "text": "The MLflow UI is one of my favorite ways to inspect experiment results.",
      "sentiment": "positive"
    },
    {
      "text": "I'm impressed by how easy it is to log metrics and artifacts with MLflow.",
      "sentiment": "positive"
    },
    {
      "text": "MLflow saved me hours of work when I needed to reproduce an old experiment.",
      "sentiment": "positive"
    },
    {
      "text": "Grrr... I really don't like when people refuse to use MLflow.",
      "sentiment": "negative"
    },
    {
      "text": "I accidentally deleted my MLflow experiment, and now I'm frustrated.",
      "sentiment": "negative"
    },
    {
      "text": "I can't believe someone stored all their experiment results in random spreadsheets instead of MLflow.",
      "sentiment": "negative"
    },
    {
      "text": "Our MLflow server was offline this morning, which completely interrupted my workflow.",
      "sentiment": "negative"
    },
    {
      "text": "I forgot to log my model to MLflow and now I have to rerun everything.",
      "sentiment": "negative"
    },
    {
      "text": "Debugging a misconfigured MLflow Tracking Server was not a fun afternoon.",
      "sentiment": "negative"
    },
    {
      "text": "MLflow is an open-source platform for managing machine learning workflows.",
      "sentiment": "neutral"
    },
    {
      "text": "Our organization uses MLflow to track experiments.",
      "sentiment": "neutral"
    },
    {
      "text": "The latest training run was logged to MLflow.",
      "sentiment": "neutral"
    },
    {
      "text": "MLflow supports experiment tracking, model management, and evaluation features.",
      "sentiment": "neutral"
    },
    {
      "text": "The data science team reviewed the metrics stored in MLflow during today's meeting.",
      "sentiment": "neutral"
    },
    {
      "text": "An MLflow Tracking Server is running in our development environment.",
      "sentiment": "neutral"
    },
    {
      "text": "The engineer opened the MLflow UI to inspect the latest experiment results.",
      "sentiment": "neutral"
    }
]

## 1: Basic Prompt Registration
In this section, we'll demonstrate how you can simply register a prompt to the MLflow prompt registry. Before we get into this, it is first important to recognize that MLflow supports two different prompt types:

1. **Text**: This is a simple string. It accepts variables delineated by double curly braces (e.g. `{{ variable_name }}`)
2. **Chat**: This simulates the beginning of a back-and-forth chat conversation. This may be helpful for things like few shot prompting.

In the cells below, we will demonstrate how you can simply register each of these respective types of prompts programatically.

### 1.1: Registering a Text Prompt

In [7]:
# Registering our summarization prompt as a text prompt
registered_summarization_prompt = mlflow.genai.register_prompt(
    name = 'summarization-prompt',
    template = SUMMARIZATION_PROMPT,
    commit_message = 'Registering the initial version of the summarization prompt'
)

print(f"Created prompt '{registered_summarization_prompt.name}' (version {registered_summarization_prompt.version})")

2026/07/03 18:19:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: summarization-prompt, version 2


Created prompt 'summarization-prompt' (version 2)


### 1.2: Registering a Chat Prompt

In [8]:
# Registering our summarization prompt as a chat prompt
registered_summarization_chat_prompt = mlflow.genai.register_prompt(
    name = 'summarization-chat-prompt',
    template = SUMMARIZATION_CHAT_PROMPT,
    commit_message = 'Registering the initial version of the summarization chat prompt'
)

print(f"Created prompt '{registered_summarization_chat_prompt.name}' (version {registered_summarization_chat_prompt.version})")

2026/07/03 18:19:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: summarization-chat-prompt, version 2


Created prompt 'summarization-chat-prompt' (version 2)


## 2: Advanced Prompt Registration
In the previous section, we registered a basic text and chat version of our summarization prompt. In this section, we'll take things a step further by demonstrating how we can apply additional configuration to each of these prompts. We'll overwrite these prompts in MLflow's prompt registry to also demonstrate how versioning works.

We'll be adding the following bits of information:

1. **Response format**: You can set the expected response format either using Pydantic or JSON. In the cell below, we will set a response format using Pydantic.
2. **Model configuration**: It is possible to set the model configuration that the prompt is intended to be used with. For our purposes, we will be making use of the OpenAI GPT-5.4 nano endpoint as we have manifested through MLflow's AI Gateway.
3. **Tags**: These are simply additional bits of metadata about the registered prompt.

In [ ]:
# Creating a response structure using Pydantic
class SummaryResponseFormat(BaseModel):
    summary: str = Field(..., description = 'Summary of the content')

### 2.1: Registering a Text Prompt the Advanced Way

In [ ]:
# Re-registering the text prompt with additional configuration
registered_summarization_prompt = mlflow.genai.register_prompt(
    name = 'summarization-prompt',
    template = SUMMARIZATION_PROMPT,
    commit_message = 'Registering the prompt with additional config information',
    response_format = SummaryResponseFormat,
    model_config = {
        'model_name': f'gateway:/{OPENAI_GPT_5_4_NANO_ENDPOINT}',
        'temperature': 0.7,
        'max_tokens': 1000
    },
    tags = {
        'author': 'dkhundley'
    }
)

print(f"Created prompt '{registered_summarization_prompt.name}' (version {registered_summarization_prompt.version})")

2026/07/03 18:39:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: summarization-prompt, version 4


Created prompt 'summarization-prompt' (version 4)


### 2.2: Registering a Chat Prompt the Advanced Way

In [12]:
# Re-registering the text prompt with additional configuration
registered_summarization_chat_prompt = mlflow.genai.register_prompt(
    name = 'summarization-chat-prompt',
    template = SUMMARIZATION_CHAT_PROMPT,
    commit_message = 'Registering the chat prompt with additional config information',
    response_format = SummaryResponseFormat,
    model_config = {
        'model_name': f'gateway:/{OPENAI_GPT_5_4_NANO_ENDPOINT}',
        'temperature': 0.7,
        'max_tokens': 1000
    },
    tags = {
        'author': 'dkhundley'
    }
)

print(f"Created prompt '{registered_summarization_chat_prompt.name}' (version {registered_summarization_chat_prompt.version})")

2026/07/03 18:41:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: summarization-chat-prompt, version 3


Created prompt 'summarization-chat-prompt' (version 3)


## 3: Loading Prompts from the Prompt Registry

### 3.1: Using the Text Prompt

### 3.2: Using the Chat Prompt

## 4: Auto-rewrite of Prompts